# Vision RAG — ColQwen2.5 Embedding + ViDoRe v3 Evaluation

**Run this notebook once per dataset.** Each run takes ~2h on T4.

1. Set `DATASET_INDEX` below (0, 1, or 2)
2. Run All
3. Download the zip at the end
4. Change `DATASET_INDEX`, Runtime → Restart, Run All again

| Index | Dataset | ~Pages | ~Time on T4 |
|-------|---------|--------|-------------|
| 0 | finance_en | 2,942 | ~2h |
| 1 | computer_science | 1,360 | ~1h |
| 2 | hr | 1,110 | ~50min |

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Pick Your Dataset

In [ ]:
#  ╔═══════════════════════════════════════════════╗
#  ║  CHANGE THIS: 0 = finance, 1 = CS, 2 = HR    ║
#  ╚═══════════════════════════════════════════════╝
DATASET_INDEX = 0

DATASETS = [
    'vidore/vidore_v3_finance_en',
    'vidore/vidore_v3_computer_science',
    'vidore/vidore_v3_hr',
]

DATASET_NAME = DATASETS[DATASET_INDEX]
DATASET_KEY = DATASET_NAME.split('vidore_v3_')[-1]

print(f'Processing: {DATASET_KEY} ({DATASET_NAME})')

## 2. Setup

In [ ]:
!nvidia-smi

In [ ]:
%%time
!pip install -q colpali-engine>=0.3.0 datasets>=3.0.0 pytrec-eval-terrier>=0.5.0

In [ ]:
import os
import torch
import numpy as np
import json
import time
from datasets import load_dataset
from PIL import Image

# All output goes here — downloaded as zip at the end
OUTPUT_DIR = f'/content/outputs/{DATASET_KEY}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'Output: {OUTPUT_DIR}')

## 3. Load ColQwen2.5

In [ ]:
%%time
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor

MODEL_NAME = 'vidore/colqwen2.5-v0.2'

model = ColQwen2_5.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda',
).eval()

processor = ColQwen2_5_Processor.from_pretrained(MODEL_NAME)

print(f'Model loaded. VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 4. Helper Functions

In [ ]:
def embed_pages(images, batch_size=2):
    """Embed page images with ColQwen2.5.
    Returns list of numpy arrays, each (n_patches, dim)."""
    all_embeddings = []
    total = len(images)

    for i in range(0, total, batch_size):
        batch_imgs = images[i:i + batch_size]
        batch = processor.process_images(batch_imgs)
        batch = {k: v.to(model.device) for k, v in batch.items()}

        with torch.no_grad():
            embeddings = model(**batch)

        for emb in embeddings:
            all_embeddings.append(emb.cpu().float().numpy())

        done = min(i + batch_size, total)
        if done % 100 < batch_size or done == total:
            elapsed = time.time() - _embed_start
            eta = (elapsed / done) * (total - done) / 60
            print(f'  {done}/{total} pages  ({elapsed/60:.1f}min elapsed, ~{eta:.0f}min remaining)')

        del batch, embeddings
        torch.cuda.empty_cache()

    return all_embeddings


def embed_queries(queries, batch_size=8):
    """Embed text queries. Returns list of (n_tokens, dim) arrays."""
    all_embeddings = []
    for i in range(0, len(queries), batch_size):
        batch = processor.process_queries(queries[i:i + batch_size])
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.no_grad():
            embeddings = model(**batch)
        for emb in embeddings:
            all_embeddings.append(emb.cpu().float().numpy())
        del batch, embeddings
        torch.cuda.empty_cache()
    print(f'  {len(all_embeddings)} queries embedded')
    return all_embeddings


def prune_l2_topk(embeddings_list, keep_fraction=0.5):
    """Keep top patches by L2 norm per page."""
    pruned = []
    for emb in embeddings_list:
        norms = np.linalg.norm(emb, axis=1)
        k = max(1, int(len(norms) * keep_fraction))
        top_idx = np.argpartition(norms, -k)[-k:]
        top_idx.sort()
        pruned.append(emb[top_idx])
    return pruned


def save_embeddings(embeddings_list, metadata, path):
    """Save as .npz + metadata JSON sidecar."""
    arrays = {f'page_{i}': emb for i, emb in enumerate(embeddings_list)}
    np.savez_compressed(path, **arrays)
    with open(path.replace('.npz', '_meta.json'), 'w') as f:
        json.dump(metadata, f)
    print(f'  Saved → {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_embeddings(path):
    """Load .npz + metadata."""
    data = np.load(path, allow_pickle=True)
    embeddings = [data[f'page_{i}'] for i in range(len(data.files))]
    with open(path.replace('.npz', '_meta.json')) as f:
        metadata = json.load(f)
    return embeddings, metadata


def maxsim_scores(query_emb, page_embeddings):
    """MaxSim: for each query token, max sim across page patches, then sum."""
    scores = np.zeros(len(page_embeddings))
    for i, page_emb in enumerate(page_embeddings):
        sims = query_emb @ page_emb.T
        scores[i] = sims.max(axis=1).sum()
    return scores


print('✓ Helpers ready')

## 5. Explore Dataset Structure

Run this first to see column names. If they differ from the defaults in the next cell, update them.

In [ ]:
ds = load_dataset(DATASET_NAME, trust_remote_code=True)
print(f'Dataset: {DATASET_NAME}')
print(f'Splits: {list(ds.keys())}\n')

for split_name, split_data in ds.items():
    print(f'--- {split_name} ({len(split_data)} rows) ---')
    print(f'  Columns: {split_data.column_names}')
    if len(split_data) > 0:
        row = split_data[0]
        for col in split_data.column_names:
            val = row[col]
            if isinstance(val, Image.Image):
                print(f'  {col}: PIL Image {val.size}')
            elif isinstance(val, str) and len(val) > 80:
                print(f'  {col}: "{val[:80]}..."')
            else:
                print(f'  {col}: {repr(val)}')
    print()

## 6. Column Config

**Check cell 5 output.** If column/split names differ, update below.

In [ ]:
# Update these if cell 5 shows different names
CORPUS_SPLIT = 'corpus'
QUERIES_SPLIT = 'queries'
QRELS_SPLIT = 'qrels'
CORPUS_ID_COL = 'corpus_id'
CORPUS_IMAGE_COL = 'image'
QUERY_ID_COL = 'query_id'
QUERY_TEXT_COL = 'query'
QREL_QUERY_COL = 'query_id'
QREL_CORPUS_COL = 'corpus_id'
QREL_SCORE_COL = 'score'

print('Config set ✓')

## 7. Embed Pages

⚠️ **This is the long cell.** ~2h for finance_en, ~1h for CS, ~50min for HR.

In [ ]:
corpus = ds[CORPUS_SPLIT]
print(f'Corpus: {len(corpus)} pages')

corpus_ids = [str(row[CORPUS_ID_COL]) for row in corpus]
corpus_images = [row[CORPUS_IMAGE_COL] for row in corpus]

print(f'\nEmbedding {len(corpus_images)} pages...')
_embed_start = time.time()
page_embeddings = embed_pages(corpus_images, batch_size=2)
embed_time = time.time() - _embed_start

print(f'\n✓ Done in {embed_time/60:.1f} min ({embed_time/len(corpus_images):.1f} sec/page)')
print(f'  Shape sample: {page_embeddings[0].shape}')

# Save immediately
page_meta = [{'corpus_id': cid, 'index': i} for i, cid in enumerate(corpus_ids)]
save_embeddings(page_embeddings, page_meta, f'{OUTPUT_DIR}/pages.npz')

## 8. Embed Queries

In [ ]:
queries = ds[QUERIES_SPLIT]
print(f'Queries: {len(queries)}')

query_ids = [str(row[QUERY_ID_COL]) for row in queries]
query_texts = [row[QUERY_TEXT_COL] for row in queries]

print(f'\nEmbedding {len(query_texts)} queries...')
query_embeddings = embed_queries(query_texts, batch_size=8)

query_meta = [{'query_id': qid, 'text': qt, 'index': i}
              for i, (qid, qt) in enumerate(zip(query_ids, query_texts))]
save_embeddings(query_embeddings, query_meta, f'{OUTPUT_DIR}/queries.npz')

## 9. DocPruner (L2-norm top 50%)

In [ ]:
print('Pruning (keep top 50% patches by L2 norm)...')
pruned = prune_l2_topk(page_embeddings, keep_fraction=0.5)

raw_patches = sum(e.shape[0] for e in page_embeddings)
pruned_patches = sum(e.shape[0] for e in pruned)
print(f'  {raw_patches} → {pruned_patches} patches ({pruned_patches/raw_patches:.1%})')

save_embeddings(pruned, page_meta, f'{OUTPUT_DIR}/pages_pruned.npz')

## 10. Save Qrels

In [ ]:
if QRELS_SPLIT in ds:
    qrels_data = ds[QRELS_SPLIT]
    qrels_list = []
    for row in qrels_data:
        entry = {}
        for k, v in dict(row).items():
            entry[k] = v if isinstance(v, (str, int, float, bool, list, dict, type(None))) else str(v)
        qrels_list.append(entry)

    with open(f'{OUTPUT_DIR}/qrels.json', 'w') as f:
        json.dump(qrels_list, f)
    print(f'Saved {len(qrels_list)} qrels')
else:
    print(f'No qrels split found. Available splits: {list(ds.keys())}')

## 11. Evaluate: MaxSim + NDCG@5

In [ ]:
import pytrec_eval

# Load pruned embeddings for scoring
page_embs, p_meta = load_embeddings(f'{OUTPUT_DIR}/pages_pruned.npz')
query_embs, q_meta = load_embeddings(f'{OUTPUT_DIR}/queries.npz')

# Load qrels
with open(f'{OUTPUT_DIR}/qrels.json') as f:
    qrels_list = json.load(f)

qrels_dict = {}
for row in qrels_list:
    qid = str(row[QREL_QUERY_COL])
    cid = str(row[QREL_CORPUS_COL])
    score = int(row[QREL_SCORE_COL])
    qrels_dict.setdefault(qid, {})[cid] = score

print(f'Scoring {len(query_embs)} queries × {len(page_embs)} pages...')
run_dict = {}
latencies = []

for qi, (q_emb, qm) in enumerate(zip(query_embs, q_meta)):
    qid = qm['query_id']
    if qid not in qrels_dict:
        continue

    t0 = time.time()
    scores = maxsim_scores(q_emb, page_embs)
    latencies.append(time.time() - t0)

    top_k = min(10, len(scores))
    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    run_dict[qid] = {p_meta[idx]['corpus_id']: float(scores[idx]) for idx in top_idx}

    if (qi + 1) % 200 == 0:
        print(f'  {qi+1}/{len(query_embs)}')

# Compute metrics
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels_dict, {'ndcg_cut_5', 'ndcg_cut_10', 'recall_5', 'recall_10', 'map'}
)
eval_results = evaluator.evaluate(run_dict)

metrics = {}
for m in ['ndcg_cut_5', 'ndcg_cut_10', 'recall_5', 'recall_10', 'map']:
    vals = [r[m] for r in eval_results.values()]
    metrics[m] = sum(vals) / len(vals) if vals else 0

lat = np.array(latencies) * 1000
metrics['retrieval_p50_ms'] = float(np.percentile(lat, 50))
metrics['retrieval_p95_ms'] = float(np.percentile(lat, 95))
metrics['retrieval_p99_ms'] = float(np.percentile(lat, 99))

print(f'\n{"=" * 50}')
print(f'  {DATASET_KEY} — Vision RAG (ColQwen2.5, pruned 50%)')
print(f'{"=" * 50}')
print(f'  NDCG@5:     {metrics["ndcg_cut_5"]:.4f}')
print(f'  NDCG@10:    {metrics["ndcg_cut_10"]:.4f}')
print(f'  Recall@5:   {metrics["recall_5"]:.4f}')
print(f'  Recall@10:  {metrics["recall_10"]:.4f}')
print(f'  MAP:        {metrics["map"]:.4f}')
print(f'  p95 latency: {metrics["retrieval_p95_ms"]:.0f}ms')
print(f'{"=" * 50}')

## 12. Save Results + Download

In [ ]:
# Save results JSON
results = {
    'dataset': DATASET_KEY,
    'model': MODEL_NAME,
    'prune_fraction': 0.5,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'n_pages': len(corpus_images),
    'n_queries': len(query_texts),
    'embed_time_sec': embed_time,
    'sec_per_page': embed_time / len(corpus_images),
    'raw_patches': raw_patches,
    'pruned_patches': pruned_patches,
    'compression_ratio': raw_patches / pruned_patches,
    **metrics,
}

with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:
# Zip and download
import shutil
from google.colab import files

zip_name = f'{DATASET_KEY}_embeddings'
shutil.make_archive(f'/content/{zip_name}', 'zip', OUTPUT_DIR)
zip_path = f'/content/{zip_name}.zip'
print(f'\nZip: {zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB)')
print(f'\nDownloading...')
files.download(zip_path)

## Done!

You downloaded `{dataset}_embeddings.zip` containing:

```
pages.npz              — raw page embeddings
pages_meta.json        — corpus_id mapping
pages_pruned.npz       — pruned (50%) page embeddings
pages_pruned_meta.json — same mapping for pruned
queries.npz            — query embeddings
queries_meta.json      — query_id + text mapping
qrels.json             — relevance judgments
results.json           — NDCG@5, Recall, latency, compression
```

**Unzip into your local project:**
```
embeddings/{dataset_key}/    ← all .npz and .json files go here
```

**Next dataset:** Change `DATASET_INDEX` at the top → Runtime → Restart and run all